# nilearn-aal-viewer — Demo

This notebook demonstrates `show_brain_viewer` on a **synthetic t-map**  
(no real patient data required).

To use your own statistical map, replace `tmap_nii` with any NIfTI  
t-map or z-map in MNI space.

In [ ]:
import sys
sys.path.insert(0, '..')   # allow import from repo root

import numpy as np
import nibabel as nib
from pathlib import Path
from nilearn import datasets

from viewer import build_aal_lookup, lut_to_json, show_brain_viewer

## 1 — Atlas path

Download AAL3v1 from:  
https://www.gin.cnrs.fr/en/tools/aal/

Then set the path below.

In [ ]:
AAL_PATH = Path('path/to/AAL3v1_1mm.nii.gz')   # <-- set this

## 2 — Build the lookup table (once per session)

In [ ]:
lut      = build_aal_lookup(aal_path=AAL_PATH, step_mm=2)
lut_json = lut_to_json(lut)
print(f'{len(lut)} voxels indexed, {len(set(lut.values()))} regions')

## 3 — Create a synthetic t-map

Gaussian noise thresholded to simulate two activation blobs.

In [ ]:
from nilearn.image import new_img_like

mni = datasets.load_mni152_template()
data = np.random.randn(*mni.shape) * 2.0

# Simulate two blobs
data[40:55, 60:70, 45:55] += 5.0   # left SMA-ish
data[60:72, 40:52, 30:40] += 4.0   # right temporal-ish

tmap_nii = new_img_like(mni, data)
print('Synthetic t-map shape:', tmap_nii.shape)

## 4 — Generate the viewer

In [ ]:
OUT_DIR = Path('../outputs')
bg_img  = datasets.load_mni152_template()

show_brain_viewer(
    img            = tmap_nii,
    threshold      = 3.5,
    title          = 'Demo contrast',
    contrast_label = 'synthetic data',
    fname          = 'viewer_demo.html',
    out_dir        = OUT_DIR,
    lut_json       = lut_json,
    step           = 2,
    bg_img         = bg_img,
    cmap           = 'cold_hot',
    opacity        = 0.8,
    height         = 520,
)

## 5 — Multiple contrasts (typical workflow)

Build the LUT once, loop over contrasts.

In [ ]:
VIEWER_CONFIGS = {
    'CB': {'thresh': 3.50, 'label': 'p<0.001 uncorrected', 'fname': 'viewer_C_vs_B.html'},
    'RB': {'thresh': 2.81, 'label': 'p<0.005 uncorrected', 'fname': 'viewer_R_vs_B.html'},
    'CR': {'thresh': 2.03, 'label': 'p<0.05 exploratory',  'fname': 'viewer_C_vs_R.html'},
}

SHORT_LABELS = {'CB': 'C > B', 'RB': 'R > B', 'CR': 'C > R'}

# Replace tmap_nii with your actual per-contrast NIfTI images
imgs_t = {'CB': tmap_nii, 'RB': tmap_nii, 'CR': tmap_nii}

for key, cfg in VIEWER_CONFIGS.items():
    show_brain_viewer(
        img            = imgs_t[key],
        threshold      = cfg['thresh'],
        title          = SHORT_LABELS[key],
        contrast_label = cfg['label'],
        fname          = cfg['fname'],
        out_dir        = OUT_DIR,
        lut_json       = lut_json,
        step           = 2,
        bg_img         = bg_img,
    )